# Chapter 9 — Indoor Propagation Modeling — equations

Standalone, runnable subset of the master `../RF_Equations.ipynb`, scoped to this chapter.
Run top-to-bottom: **Setup**, then this chapter's sections. All functions are verified against the book's worked examples.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C = 2.99792458e8        # speed of light, m/s
EPS0 = 8.8541878128e-12 # vacuum permittivity, F/m

def wavelength(f_hz):
    return C / f_hz


In [ ]:
import math
# Prerequisites from other chapters (so this notebook runs standalone):
def fspl_db(d_m, f_hz):                                   # Ch 4 (master §1)
    return 20*np.log10(4*np.pi*np.asarray(d_m, float)*f_hz / C)
def norm_cdf(x):                                          # Ch 8 (master §16)
    return 0.5*(1 + np.vectorize(math.erf)(np.asarray(x, float)/math.sqrt(2)))
def norm_ppf(p):                                          # Ch 8 (master §16, Acklam)
    a=[-3.969683028665376e1,2.209460984245205e2,-2.759285104469687e2,1.383577518672690e2,-3.066479806614716e1,2.506628277459239e0]
    b=[-5.447609879822406e1,1.615858368580409e2,-1.556989798598866e2,6.680131188771972e1,-1.328068155288572e1]
    c=[-7.784894002430293e-3,-3.223964580411365e-1,-2.400758277161838e0,-2.549732539343734e0,4.374664141464968e0,2.938163982698783e0]
    d=[7.784695709041462e-3,3.224671290700398e-1,2.445134137142996e0,3.754408661907416e0]
    def one(p):
        if p<0.02425:
            q=math.sqrt(-2*math.log(p));  return (((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5])/((((d[0]*q+d[1])*q+d[2])*q+d[3])*q+1)
        if p>1-0.02425:
            q=math.sqrt(-2*math.log(1-p));return -(((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5])/((((d[0]*q+d[1])*q+d[2])*q+d[3])*q+1)
        q=p-0.5; r=q*q; return (((((a[0]*r+a[1])*r+a[2])*r+a[3])*r+a[4])*r+a[5])*q/(((((b[0]*r+b[1])*r+b[2])*r+b[3])*r+b[4])*r+1)
    return np.vectorize(one)(np.asarray(p, float))
def shadowing_margin_db(sigma_L, coverage):               # Ch 8 (master §16)
    return norm_ppf(coverage)*sigma_L
def coherence_bandwidth_hz(rms_delay_spread_s, factor=5.0):  # Ch 8 (master §16)
    return 1.0/(factor*rms_delay_spread_s)

## 2. Log-Distance Path Loss — Ch 9.3.4

$$PL(d) = PL(d_0) + 10\,n\log_{10}\!\frac{d}{d_0} + X_\sigma$$

`n` = path-loss exponent (2 = free space, 3–5 indoors), `X_σ` = log-normal shadowing.

**Engine hook:** cheapest `PL(x,y)` layer. `n` absorbs average clutter; add walls
explicitly via the ITU/Motley–Keenan term (next).


In [ ]:
def log_distance_pl_db(d_m, n=3.0, pl_d0_db=40.0, d0_m=1.0, sigma_db=0.0, rng=None):
    d_m = np.asarray(d_m, float)
    mean = pl_d0_db + 10.0*n*np.log10(d_m/d0_m)
    if sigma_db <= 0:
        return mean
    r = rng or np.random.default_rng(0)
    return mean + r.normal(0.0, sigma_db, size=np.shape(d_m))

d = np.linspace(1, 50, 200)
plt.figure(figsize=(6,4))
for n in (2.0, 3.0, 4.0):
    plt.plot(d, log_distance_pl_db(d, n=n), label=f"n={n}")
plt.xlabel("distance (m)"); plt.ylabel("PL (dB)")
plt.title("Log-distance path loss"); plt.legend(); plt.grid(True); plt.show()


## 3. ITU Indoor Path Loss — Ch 9.3.3

$$PL = 20\log_{10} f_{MHz} + N\log_{10} d + L_f(n) - 28$$

`N` = distance power-loss coefficient (e.g. ~28 office / ~30 residential @ ~2 GHz),
`L_f(n)` = floor-penetration loss for `n` floors.

**Engine hook:** `N` and `L_f` map directly onto the Motley–Keenan distance term and the
floor-count penalty already in the path-loss layer.


In [ ]:
def itu_indoor_pl_db(d_m, f_mhz, N=28.0, Lf=0.0):
    # ITU-R P.1238 form. Lf is total floor-penetration loss (dB) for the path.
    return 20*np.log10(f_mhz) + N*np.log10(np.asarray(d_m, float)) - 28.0 + Lf

d = np.linspace(1, 50, 200)
plt.figure(figsize=(6,4))
for N, lab in ((22,"open"), (28,"office"), (30,"residential")):
    plt.plot(d, itu_indoor_pl_db(d, 2400.0, N=N), label=f"N={N} ({lab})")
plt.xlabel("distance (m)"); plt.ylabel("PL (dB)")
plt.title("ITU indoor path loss @ 2.4 GHz"); plt.legend(); plt.grid(True); plt.show()


## 17. Indoor Propagation — Ch 9  *(the deployment use case)*

The two site-general models that **are the engine's path-loss layer**. Both **verified below**
against Ex 9.1 & 9.2 (seeded `itu_indoor_pl_db` = eq 9.1, `log_distance_pl_db` = eq 9.2 — my first
blind pass, now confirmed). Indoors, deterministic models are rare (layout/materials/people change),
so statistical models fit to data are the norm.

- **ITU indoor** `PL = 20log f + N·log d + Lf(n) − 28` (eq 9.1) — a modified power law; **N = 20 ⇒
  free space**, N = 18 corridor (channeling), N = 40 through walls / around corners. Table 9.1 → N by
  band+environment; Table 9.2 → floor loss Lf(n). → `ITU_N`, `itu_floor_loss_db()`.
- **Log-distance** `PL = PL(d0) + N·log(d/d0) + Xσ` (eq 9.2) — exponent = N/10; Xσ ~ N(0,σ) shadowing
  (Ch 8). Table 9.4 → N & σ by building. *Rappaport: indoor σ ≈ 13 dB ⇒ ±26 dB (2σ) is normal.*
- **Delay spread:** indoor impulse response `h(t) = e^(−t/S)`, 0<t<tmax (S = rms delay spread) — the
  exponential profile Ch 8.4 pointed to. Table 9.3: S ~ 20–500 ns. → `indoor_impulse_response()`.

*These N / Lf / σ tables are the literal per-environment inputs for the engine's path-loss layer
(the empirical counterpart to the physics Fresnel/absorption of Ch 2).*


In [ ]:
# Table 9.1: ITU distance power-loss coefficient N  (band -> environment -> N)
ITU_N = {
    "900MHz":    {"office": 33, "commercial": 20},
    "1.2-1.3GHz":{"office": 32, "commercial": 22},
    "1.8-2GHz":  {"residential": 28, "office": 30, "commercial": 22},
    "4GHz":      {"office": 28, "commercial": 22},
    "5.2GHz":    {"office": 31},
    "60GHz":     {"office": 22, "commercial": 17},   # assumed same room
}
# N rules of thumb: 20 = free space / open area, 18 = corridor (channeling), 40 = through walls / corners

def itu_floor_loss_db(n, band="1.8-2GHz", env="office"):   # Table 9.2, Lf(n)
    if band == "1.8-2GHz":
        return {"residential": 4*n, "office": 15 + 4*(n-1), "commercial": 6 + 3*(n-1)}[env]
    if band == "900MHz" and env == "office":  return {1: 9, 2: 19, 3: 24}[n]
    if band == "5.2GHz" and env == "office":  return 16      # n = 1 only
    raise ValueError("no ITU floor-loss data for that band/env")

# Example 9.1: 5.2 GHz office, 100 m, N=31
pl_same  = itu_indoor_pl_db(100, 5200, N=31, Lf=0)
pl_floor = itu_indoor_pl_db(100, 5200, N=31, Lf=itu_floor_loss_db(1, "5.2GHz"))
print(f"Ex 9.1: same-floor PL = {pl_same:.0f} dB (book 108), +1 floor = {pl_floor:.0f} dB "
      f"(+{itu_floor_loss_db(1,'5.2GHz')} dB floor loss)")


In [ ]:
# Table 9.4: log-distance N (dB/decade) and shadowing sigma (dB)
LOGDIST_PARAMS = {   # building: (freq_MHz, N, sigma_dB)
    "retail":           (914, 22, 8.7),  "grocery":         (914, 18, 5.2),
    "office_hard_part": (1500, 30, 7.0), "office_soft_900": (900, 24, 9.6),
    "office_soft_1900": (1900, 26, 14.1),"textile_1300":    (1300, 20, 3.0),
    "paper_cereals":    (1300, 18, 6.0), "metalworking":    (1300, 16, 5.8),
}

# Example 9.2: 1.5 GHz office (hard partition), 100 m, 95% coverage -> N=30, sigma=7.0
pl_d0  = fspl_db(1, 1.5e9)                       # reference free-space loss at 1 m
Xs     = shadowing_margin_db(7.0, 0.95)          # z*sigma, z(95%) = 1.645
median = log_distance_pl_db(100, n=30/10, pl_d0_db=pl_d0)   # exponent = N/10 = 3.0
print(f"Ex 9.2: PL(1m)={pl_d0:.0f} dB, Xs(95%)={Xs:.1f} dB (book 11.5), "
      f"total={median+Xs:.1f} dB (book 107.5); FSL@100m={fspl_db(100,1.5e9):.0f} dB (book 76)")


In [ ]:
# Table 9.3: rms delay spread S (ns) -- often / median / rarely
INDOOR_DELAY_SPREAD_NS = {
    "1.9GHz_residential": (20, 70, 150), "1.9GHz_office":     (35, 100, 460),
    "1.9GHz_commercial":  (55, 150, 500), "5.2GHz_office":    (45, 75, 150),
}
def indoor_impulse_response(t_s, S_s, tmax_s):
    # ITU indoor channel impulse response: h(t) = exp(-t/S) for 0 < t < tmax, else 0.
    t = np.asarray(t_s, float)
    return np.where((t > 0) & (t < tmax_s), np.exp(-t/S_s), 0.0)

t = np.linspace(-0.5e-6, 3e-6, 400)
plt.figure(figsize=(6,3.5))
plt.plot(t*1e6, indoor_impulse_response(t, 1e-6, 2.5e-6))
plt.xlabel("t (us)"); plt.ylabel("h(t)"); plt.title("ITU indoor impulse response (S=1 us, tmax=2.5 us)")
plt.grid(True); plt.show()
# a median office delay spread -> coherence bandwidth (Ch 8.4)
S = 100e-9
print(f"median 1.9 GHz office S=100 ns -> coherence BW ~ {coherence_bandwidth_hz(S)/1e6:.1f} MHz "
      f"(a 20 MHz Wi-Fi channel is wider -> frequency-selective, needs OFDM)")


## 18. Chapter 9 Exercises — worked with the encoded functions

Solving the end-of-chapter problems with the notebook's own functions — a working-tool check that
also exercises §16–§17. Where the book under-specifies a parameter, the assumption is stated.


In [ ]:
# Ex 9-1: median ITU-indoor PL, 1.9 GHz office, 100 m  (N=30 office, Table 9.1; Lf=0 same floor)
pl1 = itu_indoor_pl_db(100, 1900, N=30, Lf=0)
print(f"Ex 9-1: ITU median PL = {pl1:.1f} dB")

# Ex 9-2: PL vs probability of occurrence. ITU gives no sigma, so borrow a typical office sigma = 8 dB.
sigma = 8.0
p = np.linspace(0.01, 0.99, 200)
plt.figure(figsize=(6,4))
plt.plot(p*100, pl1 + norm_ppf(p)*sigma)
plt.xlabel("probability that PL <= value  (%)"); plt.ylabel("path loss (dB)")
plt.title(f"Ex 9-2: ITU median {pl1:.0f} dB, log-normal sigma={sigma} dB"); plt.grid(True); plt.show()


In [ ]:
# Ex 9-3: log-distance, 1.9 GHz office SOFT partition, 98% coverage at 100 m (Table 9.4: N=26, sigma=14.1)
N3, sig3 = 26, 14.1
pl_d0_3  = fspl_db(1, 1.9e9)
median3  = log_distance_pl_db(100, n=N3/10, pl_d0_db=pl_d0_3)
Xs3      = shadowing_margin_db(sig3, 0.98)
print(f"Ex 9-3: PL(1m)={pl_d0_3:.1f}, median={median3:.1f}, Xs(98%)={Xs3:.1f} -> total = {median3+Xs3:.1f} dB")

# Ex 9-4: log-distance, 900 MHz office HARD partition, 38 m, min/max at 99% probability.
# Table 9.4's hard-partition entry is 1500 MHz (N=30, sigma=7.0); used here for the 900 MHz office-hard case.
N4, sig4  = 30, 7.0
pl_d0_4   = fspl_db(1, 900e6)
median4   = log_distance_pl_db(38, n=N4/10, pl_d0_db=pl_d0_4)
half      = norm_ppf(0.995)*sig4                      # central 99% -> +/- z(0.995)*sigma
print(f"Ex 9-4: median={median4:.1f} dB; central-99% range = [{median4-half:.1f}, {median4+half:.1f}] dB (+/-{half:.1f})")


In [ ]:
# Ex 9-5: highest symbol rate with NO equalizer (channel must be flat) from Table 9.3 delay spreads.
# No equalizer -> signal BW <= coherence BW (equivalently symbol period >> delay spread).
# Use the LARGEST tabulated rms delay spread (worst case) to be safe: commercial 1.9 GHz "rarely" = 500 ns.
S_worst  = 500e-9
Bc       = coherence_bandwidth_hz(S_worst)           # 1/(5 S)
Rs_10pct = 0.1/S_worst                               # symbol >= 10x delay spread
print(f"Ex 9-5: worst-case S={S_worst*1e9:.0f} ns -> Bc={Bc/1e3:.0f} kHz (Rs <= {Bc/1e3:.0f} ksps), "
      f"or 10%-of-symbol rule Rs <= {Rs_10pct/1e3:.0f} ksps")
print("        Reasoning: without equalization the channel must appear flat, so the symbol period must be")
print("        >> the delay spread; using the largest tabulated S gives the safe (conservative) rate.")
